Peacock Rides: end-to-end control on Android and iPhone

What the agent does

The user speaks. The agent sees the ride app, decides, taps, and types. It asks before booking: “AC cab, Taramani to Park Town. Ola is ₹150, Uber is ₹188, Rapido has no AC cab. Ola is cheapest. Shall I book Ola?” Only a spoken yes books. Cancel needs its own spoken yes.

Payment is not an agent job, The user pays later

There is no account or API with Ola, Uber, or Rapido. The user is already logged in on that phone.

What research allows

An App Store app, a Play Store app, Shortcuts, Siri, and screen recording cannot do this. Shortcuts can open an app only when that app publishes an action. None of the three publish booking, chat, or cancel to other apps. Screen recording can show pixels and cannot tap. Xcode UI tests can tap an app you are developing. They are not a way for a store app to drive Uber.

The two paths that can see and operate another app on the user’s own phone:





Android, on the phone. An AccessibilityService the user turns on. It reads the UI tree and sends taps, text, and scrolls. dispatchGesture is the supported way to tap coordinates. A normal app cannot tap other apps any other way. This matches direct install on the user’s Android phone. Play still forbids an accessibility service that plans and acts on its own, so this build is not a Play listing. On Android 13 and later, a file-installed app is blocked until the user opens the app’s system page and allows restricted settings, then turns the service on. Peacock’s setup screen leads them through that. The service is limited to com.ubercab, com.olacabs.customer, and com.rapido.passenger.



iPhone, only with Developer Mode. No iOS app the user downloads can read or tap Ola, Uber, or Rapido. End-to-end control is a driver attached to that personal iPhone:





iOS 27 and later: device-automation tooling can use Apple’s built-in developer services after Developer Mode is on and the developer disk image is mounted. Nothing extra has to be signed onto the phone. This still requires a paired host (a Mac, or a Linux machine with the phone’s pairing). It is not an on-phone API an app can call by itself.



iOS 16 through 26: the same control is a signed XCUITest runner (WebDriverAgent, maintained for Appium). It exposes the accessibility tree and accepts tap and type while the test session is running. A free Apple ID signs it for 7 days and a few devices. A paid developer account signs it for up to a year, and the phone’s UDID must be on that account. The phone also needs Settings, Developer, Enable UI Automation. The runner is not an App Store app, TestFlight does not distribute it for this, and it dies when the session dies.



Both iPhone modes can launch another app by bundle id: Uber com.ubercab.UberClient, Ola olacabs.OlaCabs, Rapido bike.rapido.customer. Confirm Ola’s id on the device before locking it in. Third-party screens often have no stable ids, hidden web views, and blocked screenshots. The tree is the source of truth. Vision fills in only when the tree has no text.

Jailbreaks, private input injection, and hiding a store listing are not part of the build. Provider terms can still restrict the user’s own Ola, Uber, or Rapido account. The setup screen says that.

The iPhone control channel has no login of its own. It stays on the paired host’s loopback. It is never opened to Wi-Fi for other machines.  



One loop, two drivers

Same tools on both phones: observe, tap, type, scroll, open_app, remember. No payment tool.





Observe the current tree. Screenshot only if the tree is empty.



Understand the screen: home, route, fare list, confirm, trip, chat, cancel, or payment.



Plan one action, or ask if pickup, drop, class, or permission is missing.



Act with one tap, one text entry, or one scroll.



Verify the tree changed. One fallback, then say what failed and stop that step.

On-trip reads come from the trip screen, then memory, so a repeated “what is the OTP?” does not wander the app. Call taps the in-app call control. Hurry opens the in-app chat, types a short line, and sends it. Cancel asks, then taps the app’s cancel path and checks the trip is gone.

Login, CAPTCHA, a system permission, and any payment secret are walls. The agent describes them and waits.

Build shape





agent: the loop, the permission gate, and the payment wall. main.py becomes this loop. It does not know which phone it is driving.



apps/android: voice UI plus the accessibility driver. No map.



apps/ios-bridge: the paired-host driver. It picks developer services on iOS 27 and later, and the signed runner on older iOS. The voice session can run on the host next to the phone.

peacock_rides_prototype.html is only a tone and language reference.

Build order





Android driver. On one direct-installed phone, pass restricted settings, read a screen, tap, type, and return to the same ride app.



iPhone spike. One personal iPhone. Record the iOS version. If it is 27 or later, try the built-in developer services first. Otherwise sign the test runner and renew on the real expiry (7 days or 1 year). For each of Ola, Uber, and Rapido, write down whether the fare, ETA, OTP, call, chat, and cancel controls are in the tree.



Shared screen model. Fare rows and trip facts from the tree. Vision only for empty nodes. Ignore the map.



Quote, ask, book. Three apps, spoken yes, booking taps, stop before payment. Verify the booked trip is on screen.



On-trip. ETA, distance, OTP, call, hurry message, cancel after a second yes.



Stops. Login, CAPTCHA, payment secret, a dead iPhone session, and a tap that changed nothing all end with a spoken reason.

